In [1]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy import signal

/home/sj/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
fs = 4

Imported reshape function as it will be needed to make row-wise descriptives

In [3]:
#Now we make the reshape function
#4 parameters, your dataframe, your window size or how many hz of rows should be in each row, purity for how much each row can tolerate label confounsion, and stride size for how much we should overlap
#By default this is every 10 seconds of data
def reshape_function(df, name = 'EDA', window_size = 60, stride_size = 15, hertz=8, purity_threshold=0.8):
    window_row = window_size*hertz
    stride_row = stride_size*hertz
    #Get the subject names
    subjects = sorted(
        list(set(col.split("_")[0] for col in df.columns if "_" in col))
    )
    #Store the chunks of data while we're looping across the time span
    all_window = []
    for i in subjects:
        signal_col = f'{i}_{name}'
        label_col = f'{i}_Label'
        #Kill NAs in individual sequence
        individual_df = df[[signal_col,label_col]].dropna()
        total_rows = len(individual_df)
        #For each subject we take a chunk of their data by our determined window size then stack onto a list (in this case "all window")
        #Chunk is determined through standard range check then we check the purity threshold if we keep the chunk, based on birken's paper design
        for j in range(0,total_rows-window_row+1, stride_row):
            end = j + window_row
            signal_window = individual_df[signal_col].iloc[j:end].values
            label_window = individual_df[label_col].iloc[j:end].values

            # Determine the majority label and calculate if there exists a supermajority in the chunk to determine if kept
            value_counts = pd.Series(label_window).value_counts()
            majority_label = value_counts.index[0]
            majority_count = value_counts.iloc[0]

            purity = majority_count / window_row

            # Retain only windows meeting or exceeding the requested purity ratio
            if purity >= purity_threshold:
                row_data = [i, majority_label] + list(signal_window)
                all_window.append(row_data)

    feature_cols = [f"sample_{i}" for i in range(window_row)]
    columns = ["subject_id", "label"] + feature_cols

    return pd.DataFrame(all_window, columns=columns)

    





I also want visualization boxplots for each feature set similar to Zhu lili's breakdown

Import tonic and phasic files

In [4]:
tonic = pd.read_csv("../Data/wesad_eda/tonic_signal_nk.csv")
phasic = pd.read_csv("../Data/wesad_eda/phasic_signal_nk.csv")
eda = pd.read_csv("../Data/wesad_eda/filtered_eda_signals.csv")

In [5]:
print(tonic.shape)
print(phasic.shape)
print(eda.shape)

(20892, 30)
(20892, 30)
(20892, 30)


This section is just applying the reshape function and verifying the outputs

In [6]:
eda.head()

,S15_EDA,S15_Label,S7_EDA,S7_Label,S3_EDA,S3_Label,S2_EDA,S2_Label,S14_EDA,S14_Label,...,S4_EDA,S4_Label,S13_EDA,S13_Label,S5_EDA,S5_Label,S9_EDA,S9_Label,S6_EDA,S6_Label
0,0.000000,0.0,0.000000,0.0,0.016044,0.0,0.097471,0.0,0.011914,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.037751,0.0,0.033865,0
1,0.073305,0.0,0.485077,0.0,0.178527,0.0,0.492093,0.0,0.175268,0.0,...,0.031140,0.0,0.366758,0.0,0.065186,0.0,0.144301,0.0,0.199618,0
2,0.116746,0.0,0.854691,0.0,0.319746,0.0,0.769926,0.0,0.278055,0.0,...,0.053885,0.0,0.628746,0.0,0.109112,0.0,0.224244,0.0,0.323051,0
3,0.083819,0.0,0.729083,0.0,0.296441,0.0,0.620082,0.0,0.191334,0.0,...,0.044419,0.0,0.511660,0.0,0.085603,0.0,0.197046,0.0,0.275535,0
4,0.062718,0.0,0.609167,0.0,0.256728,0.0,0.491979,0.0,0.120382,0.0,...,0.035439,0.0,0.421939,0.0,0.063158,0.0,0.172040,0.0,0.231554,0


In [7]:
tonic.head()

,S15_Tonic,S15_Label,S7_Tonic,S7_Label,S3_Tonic,S3_Label,S2_Tonic,S2_Label,S14_Tonic,S14_Label,...,S4_Tonic,S4_Label,S13_Tonic,S13_Label,S5_Tonic,S5_Label,S9_Tonic,S9_Label,S6_Tonic,S6_Label
0,0.003759,0.0,-0.008207,0.0,0.030668,0.0,0.131371,0.0,0.029557,0.0,...,0.047935,0.0,0.011682,0.0,0.002346,0.0,0.059078,0.0,0.041740,0
1,0.006247,0.0,0.011484,0.0,0.038976,0.0,0.145760,0.0,0.035674,0.0,...,0.051862,0.0,0.026645,0.0,0.004534,0.0,0.064246,0.0,0.048556,0
2,0.008738,0.0,0.031135,0.0,0.047270,0.0,0.160146,0.0,0.041805,0.0,...,0.055849,0.0,0.041600,0.0,0.006744,0.0,0.069415,0.0,0.055361,0
3,0.011228,0.0,0.050718,0.0,0.055537,0.0,0.174509,0.0,0.047941,0.0,...,0.059892,0.0,0.056527,0.0,0.008973,0.0,0.074577,0.0,0.062145,0
4,0.013713,0.0,0.070205,0.0,0.063767,0.0,0.188830,0.0,0.054075,0.0,...,0.063989,0.0,0.071406,0.0,0.011220,0.0,0.079725,0.0,0.068901,0


In [8]:
tonic_reshaped = reshape_function(tonic, name='Tonic')

In [9]:
tonic_reshaped.shape

(2106, 482)

In [10]:
tonic_reshaped.head()

,subject_id,label,sample_0,sample_1,sample_2,sample_3,sample_4,sample_5,sample_6,sample_7,...,sample_470,sample_471,sample_472,sample_473,sample_474,sample_475,sample_476,sample_477,sample_478,sample_479
0,S10,1.0,0.090853,0.090993,0.091133,0.091271,0.091407,0.091541,0.091673,0.091803,...,0.090849,0.090853,0.090858,0.090862,0.090866,0.090871,0.090875,0.090879,0.090883,0.090887
1,S10,1.0,0.104068,0.103945,0.103815,0.103678,0.103534,0.103383,0.103226,0.103062,...,0.089841,0.089832,0.089824,0.089816,0.089808,0.089801,0.089794,0.089788,0.089782,0.089777
2,S10,1.0,0.089977,0.089986,0.089995,0.090005,0.090015,0.090025,0.090036,0.090048,...,0.090789,0.090811,0.090834,0.090856,0.090879,0.090902,0.090925,0.090948,0.090972,0.090996
3,S10,1.0,0.090815,0.090826,0.090838,0.090849,0.090860,0.090870,0.090881,0.090891,...,0.094082,0.094115,0.094149,0.094182,0.094216,0.094249,0.094283,0.094316,0.094349,0.094383
4,S10,1.0,0.090891,0.090894,0.090898,0.090901,0.090904,0.090907,0.090910,0.090912,...,0.094658,0.094625,0.094592,0.094559,0.094525,0.094491,0.094456,0.094422,0.094387,0.094352


Tonic descriptives row-wise for max and mean

In [11]:
def eda_feature_extraction(row, sampling_rate=fs):
    feature_names = [
        "phasic_peak_count",
        "phasic_peak_rate",
        "phasic_mean_amplitude",
        "phasic_max_amplitude",
        "phasic_sum_amplitude",
        "phasic_mean_rise_time",
        "phasic_std_rise_time",
        "phasic_mean_recovery_time",
        "phasic_mean_onset_time",
        "phasic_mean_offset_time",
        "phasic_shape_symmetry",  # New: Ratio of rise to recovery
        "eda_symp_power"          # New: Spectral power band
    ]

    try:
        # 1. Frequency Domain Extraction (Spectral Power)
        # Calculates power in the typical EDA sympathetic band (0.045 - 0.25 Hz)
        try:
            # Adjust window segment size for Welch's method based on row length
            nperseg = min(len(row), 256) 
            freqs, psd = signal.welch(row, fs=sampling_rate, nperseg=nperseg)
            symp_band = (freqs >= 0.045) & (freqs <= 0.25)
            eda_symp_power = np.trapz(psd[symp_band], freqs[symp_band])
        except Exception:
            eda_symp_power = np.nan

        # 2. Time Domain & Phasic Extraction
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)

        peaks = info["SCR_Peaks"]
        onsets = info["SCR_Onsets"]
        recovery = info["SCR_Recovery"]
        amplitudes = info["SCR_Amplitude"]

        peak_count = len(peaks)
        duration_minutes = (len(row) / sampling_rate) / 60

        if peak_count == 0:
            return pd.Series(
                [0, 0.0] + [np.nan] * 8 + [np.nan, eda_symp_power],
                index=feature_names,
            )

        # Convert sample indices to time durations (in seconds)
        rise_times = (peaks - onsets) / sampling_rate
        recovery_times = (recovery - peaks) / sampling_rate

        mean_rise = np.nanmean(rise_times)
        mean_recovery = np.nanmean(recovery_times)

        # 3. Phasic Shape Morphometrics
        # Ratio of rise time to recovery time (measures morphological asymmetry)
        shape_symmetry = mean_rise / mean_recovery if mean_recovery > 0 else np.nan

        onset_times = onsets / sampling_rate
        offset_times = recovery / sampling_rate 
        peak_rate = peak_count / duration_minutes

        return pd.Series(
            [
                peak_count,
                peak_rate,
                np.nanmean(amplitudes),
                np.nanmax(amplitudes),
                np.nansum(amplitudes),
                mean_rise,
                np.nanstd(rise_times),
                mean_recovery,
                np.nanmean(onset_times),
                np.nanmean(offset_times),
                shape_symmetry,
                eda_symp_power
            ],
            index=feature_names,
        )

    except Exception:
        return pd.Series([np.nan] * len(feature_names), index=feature_names)

In [12]:
# intend to apply this function to phasic reshaped
def rise_time_std(row, sampling_rate = fs):
    try:
        #Uses neurokits peak detection to get the onsets then convert it to a time scale and calculate the standard deviation
        signals, info = nk.eda_peaks(row, sampling_rate=sampling_rate)
        # Calculate peak count
        peaks = info["SCR_Peaks"]
        peak_count = len(peaks)
        rise_times = (info["SCR_Peaks"] - info["SCR_Onsets"]) / sampling_rate
        rise_time_std = np.nanstd(rise_times)
        return pd.Series(
                    [peak_count, rise_time_std],
                    index=["phasic_peak_count", "phasic_std_rise_time"],
                )

    #There is a lot of rows with no peaks at all, need to decide how to handle the NAs where the std depreciates thereof
    except Exception:
        return np.nan

In [13]:
def sig2features(sig, sig_name):
    """
    sig is a signal, like tonic_reshaped
    sig_name is a string, like "tonic"
    #feature_set is a list, like [min, max]
    """
    # only want the signals, no "subject_id" and "label" columns
    sample_cols = [col for col in sig.columns if col.startswith('sample_')]
    sig_descriptives = pd.DataFrame()
    # Bring Labels into descriptives to the database
    sig_descriptives['label'] = sig['label']
    # Bring subject ID over to descriptives
    if 'subject_id' in sig.columns:
        sig_descriptives['subject_id'] = sig['subject_id']
    # Descriptives
    sig_descriptives[f"{sig_name}_mean"] = sig[sample_cols].mean(axis=1)
    sig_descriptives[f"{sig_name}_max"]  = sig[sample_cols].max(axis=1)
    sig_descriptives[f"{sig_name}_std"]  = sig[sample_cols].std(axis=1)
    sig_descriptives[f"{sig_name}_skew"]  = sig[sample_cols].skew(axis=1)
    sig_descriptives[f"{sig_name}_norm1"]  = sig[sample_cols].abs().sum(axis=1)
    sig_descriptives[f"{sig_name}_median"] = sig[sample_cols].median(axis=1)
    sig_descriptives[f"{sig_name}_min"] = sig[sample_cols].min(axis=1)
    sig_descriptives[f"{sig_name}_range"] = sig[sample_cols].max(axis=1) - sig[sample_cols].min(axis=1)
    sig_descriptives[f"{sig_name}_mean_slope"] = (sig[sample_cols].diff(axis=1).mean(axis=1))
    sig_descriptives[f"{sig_name}_kurtosis"] = sig[sample_cols].kurtosis(axis=1)
    sig_descriptives[f"{sig_name}_sampen"] = sig[sample_cols].apply(
    lambda row: nk.entropy_sample(row.dropna())[0] if not row.dropna().empty else np.nan, 
    axis=1)
    sig_descriptives[f"{sig_name}_apen"] = sig[sample_cols].apply(
    lambda row: nk.entropy_approximate(row.dropna())[0] if not row.dropna().empty else np.nan, 
    axis=1)



    if sig_name == 'phasic':
        # Define the full list of features returned by our function
        phasic_cols = [
            "phasic_peak_count",
            "phasic_peak_rate",
            "phasic_mean_amplitude",
            "phasic_max_amplitude",
            "phasic_sum_amplitude",
            "phasic_mean_rise_time",
            "phasic_std_rise_time",
            "phasic_mean_recovery_time",
            "phasic_mean_onset_time",
            "phasic_mean_offset_time",
            "phasic_shape_symmetry",
            "eda_symp_power"
        ]
        
        # Assign the apply output to all 12 columns
        sig_descriptives[phasic_cols] = sig[sample_cols].apply(eda_feature_extraction, axis=1)
        
    return sig_descriptives

In [14]:
tonic_descriptives = sig2features(tonic_reshaped, 'tonic')
tonic_descriptives.head()

,label,subject_id,tonic_mean,tonic_max,tonic_std,tonic_skew,tonic_norm1,tonic_median,tonic_min,tonic_range,tonic_mean_slope,tonic_kurtosis,tonic_sampen,tonic_apen
0,1.0,S10,0.093515,0.104885,0.004585,1.346395,44.887122,0.090871,0.089936,0.014949,7.126101e-08,0.384893,0.011802,0.046804
1,1.0,S10,0.091567,0.104068,0.002905,2.917518,43.952038,0.090752,0.089777,0.014291,-2.983570e-05,7.681392,0.008420,0.007422
2,1.0,S10,0.090469,0.090996,0.000387,-0.466725,43.424990,0.090508,0.089741,0.001254,2.125572e-06,-1.088915,0.061261,0.067508
3,1.0,S10,0.091017,0.094383,0.001100,1.338275,43.688224,0.090815,0.089741,0.004641,7.448993e-06,1.204175,0.020953,0.017489
4,1.0,S10,0.092108,0.095704,0.002109,0.520795,44.211885,0.091008,0.089741,0.005963,7.225895e-06,-1.317859,0.019186,0.016846


Phasic descriptives row-wise for mean, peak counts and rise time standard deviation

In [15]:
# Test example before I implemented on full dataframe the rise time standard deviation descriptive
phasic_reshape = reshape_function(phasic, name='Phasic')
row_example = phasic_reshape.iloc[0:1,2:]
signals, info = nk.eda_peaks(np.array(row_example).flatten(), sampling_rate=fs)
# Calculate rise times in seconds
rise_times_sec = (info["SCR_Peaks"] - info["SCR_Onsets"]) / fs
np.nanstd(rise_times_sec)


np.float64(0.36646626128629767)

In [16]:

phasic_descriptives = sig2features(phasic_reshape, 'phasic')
phasic_descriptives.head()

/tmp/ipykernel_713550/1131894471.py:51: RuntimeWarning: Mean of empty slice
  mean_recovery = np.nanmean(recovery_times)
/tmp/ipykernel_713550/1131894471.py:72: RuntimeWarning: Mean of empty slice
  np.nanmean(offset_times),


,label,subject_id,phasic_mean,phasic_max,phasic_std,phasic_skew,phasic_norm1,phasic_median,phasic_min,phasic_range,...,phasic_mean_amplitude,phasic_max_amplitude,phasic_sum_amplitude,phasic_mean_rise_time,phasic_std_rise_time,phasic_mean_recovery_time,phasic_mean_onset_time,phasic_mean_offset_time,phasic_shape_symmetry,eda_symp_power
0,1.0,S10,0.000174,0.054931,0.007321,3.796079,1.547716,-0.000224,-0.016724,0.071655,...,0.023564,0.067419,0.259207,0.704545,0.366466,0.900000,15.772727,32.750000,0.782828,NaN
1,1.0,S10,-0.000214,0.006591,0.001459,-1.300324,0.400138,-0.000099,-0.006700,0.013291,...,0.001498,0.007155,0.052434,0.707143,0.313310,1.156250,59.228571,62.296875,0.611583,NaN
2,1.0,S10,0.000031,0.006591,0.000810,3.083956,0.233709,-0.000022,-0.001668,0.008259,...,0.001315,0.007155,0.051289,0.641026,0.258167,1.458333,57.692308,61.472222,0.439560,NaN
3,1.0,S10,0.000023,0.010082,0.001210,4.113657,0.307119,-0.000061,-0.002303,0.012385,...,0.002405,0.011035,0.052910,0.693182,0.260374,1.339286,62.352273,57.232143,0.517576,NaN
4,1.0,S10,0.000026,0.010082,0.001100,4.421792,0.294958,0.000028,-0.002303,0.012385,...,0.002185,0.011035,0.050264,0.717391,0.258595,1.519231,61.880435,57.788462,0.472207,NaN


Create and save file descriptive file

In [17]:
eda_reshaped = reshape_function(eda, name='EDA')
eda_descriptives = sig2features(eda_reshaped, 'eda')
eda_descriptives.head()

,label,subject_id,eda_mean,eda_max,eda_std,eda_skew,eda_norm1,eda_median,eda_min,eda_range,eda_mean_slope,eda_kurtosis,eda_sampen,eda_apen
0,1.0,S10,0.093689,0.158848,0.009054,3.665468,44.970615,0.090649,0.075917,0.082931,6.548994e-06,16.612144,0.094545,0.229623
1,1.0,S10,0.091353,0.105202,0.003022,3.391384,43.849551,0.090616,0.088191,0.017010,-3.189322e-05,11.202429,0.399931,0.527375
2,1.0,S10,0.090500,0.097583,0.000907,2.661236,43.440095,0.090448,0.088669,0.008914,6.757988e-07,14.785969,1.312522,1.186124
3,1.0,S10,0.091040,0.101435,0.001647,2.536093,43.699344,0.090673,0.088669,0.012766,9.640598e-06,9.409743,0.780159,0.827954
4,1.0,S10,0.092134,0.101435,0.002414,0.802010,44.224514,0.091061,0.088669,0.012766,7.051074e-06,-0.237704,0.544766,0.623854


In [18]:
feature_dataframe = pd.concat(
    [eda_descriptives,
     phasic_descriptives.drop(columns=["subject_id", "label"]), 
     tonic_descriptives.drop(columns=["subject_id", "label"]),
     ],
    axis=1,
)
feature_dataframe.head()

,label,subject_id,eda_mean,eda_max,eda_std,eda_skew,eda_norm1,eda_median,eda_min,eda_range,...,tonic_std,tonic_skew,tonic_norm1,tonic_median,tonic_min,tonic_range,tonic_mean_slope,tonic_kurtosis,tonic_sampen,tonic_apen
0,1.0,S10,0.093689,0.158848,0.009054,3.665468,44.970615,0.090649,0.075917,0.082931,...,0.004585,1.346395,44.887122,0.090871,0.089936,0.014949,7.126101e-08,0.384893,0.011802,0.046804
1,1.0,S10,0.091353,0.105202,0.003022,3.391384,43.849551,0.090616,0.088191,0.017010,...,0.002905,2.917518,43.952038,0.090752,0.089777,0.014291,-2.983570e-05,7.681392,0.008420,0.007422
2,1.0,S10,0.090500,0.097583,0.000907,2.661236,43.440095,0.090448,0.088669,0.008914,...,0.000387,-0.466725,43.424990,0.090508,0.089741,0.001254,2.125572e-06,-1.088915,0.061261,0.067508
3,1.0,S10,0.091040,0.101435,0.001647,2.536093,43.699344,0.090673,0.088669,0.012766,...,0.001100,1.338275,43.688224,0.090815,0.089741,0.004641,7.448993e-06,1.204175,0.020953,0.017489
4,1.0,S10,0.092134,0.101435,0.002414,0.802010,44.224514,0.091061,0.088669,0.012766,...,0.002109,0.520795,44.211885,0.091008,0.089741,0.005963,7.225895e-06,-1.317859,0.019186,0.016846


In [19]:
feature_dataframe.shape

(2106, 50)

In [20]:
feature_dataframe.to_csv("../Data/wesad_eda/feature_dataframe_nk.csv", index=False)
